# 08 — Enrichment A/B: does weather / alerts improve the delay model?

Answers whether the weather dimension (`05c`) and the dual-grain alert
feature tables (`05b`) actually improve delay prediction over the pristine
notebook 07 baseline. **Notebook 07 is never imported or modified** — this
notebook is additive-only and uses `enrich.py` (in this same directory) to
reproduce 07's load/filter/split logic verbatim, so all four variants below
train on exactly the same rows and exactly the same temporal split.

Four variants, same `XGBRegressor` params as 07
(`enable_categorical=True, tree_method='hist', random_state=42, n_jobs=-1`,
no tuning):

1. `baseline` — exactly 07's 8-column feature set
2. `+weather` — baseline + `weather/era5/hourly.parquet` columns
3. `+alerts` — baseline + route-level and stop-level alert columns
4. `+both` — baseline + weather + alerts

Precondition already verified before this notebook was written:
`s3://<bucket>/weather/era5/hourly.parquet` exists — 504 rows, full
2026-06-28..2026-07-18 coverage (21 dates x 24h).

In [1]:
# Cell 1 — Environment setup + import shared prep module
import sys
import json
import datetime
import subprocess

try:
    import xgboost as xgb
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'xgboost', '-q'], check=True)
    import xgboost as xgb

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '.')
import enrich

S3_BUCKET, fs = enrich.get_env()
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
print(f'xgboost: {xgb.__version__}')

Loaded .env from: /Users/proteeksanyal/Desktop/Learning/Transit-AI/.env
S3_BUCKET: seq-transit-ai-data-ps
xgboost: 2.1.4


## Load + filter + split (reproduces notebook 07 Cells 2-6 verbatim via `enrich.py`)

In [2]:
# Cell 2 — Load, leakage-filter, re-derive time features, build feature matrix
X_base, y, scheduled_arrival_dt, snapshot_timestamp, feature_cols, ML_RUN_DATE = \
    enrich.load_prepared(S3_BUCKET, fs)

_latest.json found -> latest_run=2026-07-17


Loaded 20,407,431 rows x 14 cols from s3://seq-transit-ai-data-ps/ml_features/v0_feature_snapshot/run_date=2026-07-17/
Manifest row_count: 20,407,431 | Loaded: 20,407,431 -> MATCH


Leakage filter: dropped 1,684,907 / 20,407,431 rows (8.26%) — post-arrival captures


Dropped 525,310 rows with null delay_minutes (no observed label)


Feature columns (8): ['route_id', 'stop_id', 'mode', 'stop_sequence', 'hour_of_day', 'day_of_week', 'is_weekend', 'is_peak']


In [3]:
# Cell 3 — Temporal split (verbatim boundary-date logic from 07 Cell 6)
train_mask, test_mask, boundary_date = enrich.temporal_split(X_base)
y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

Boundary date (test = this date onward): 2026-07-15


Train: 15,431,112 rows (84.8%), dates 2026-06-29 .. 2026-07-14


Test:  2,766,102 rows (15.2%), dates 2026-07-15 .. 2026-07-16


## Attach weather and alerts

Both joins use `scheduled_arrival_dt` (the time being predicted for), not
`snapshot_timestamp` (capture time) — per spec. The alert join is measured
for a specific leakage risk immediately below before it's used for
training.

In [4]:
# Cell 4 — Attach weather (joined on scheduled_arrival_dt)
X_weather_full, weather_match_rate = enrich.attach_weather(X_base, scheduled_arrival_dt, S3_BUCKET, fs)

Weather join match rate: 100.00%


In [5]:
# Cell 5 — Attach alerts (joined on snapshot_timestamp, prediction time, to avoid look-ahead leakage); route and stop kept separate
X_alerts_full, route_match_rate, stop_match_rate = enrich.attach_alerts(X_base, scheduled_arrival_dt, snapshot_timestamp, S3_BUCKET, fs)

/Users/proteeksanyal/Desktop/Learning/Transit-AI/notebooks/./enrich.py:262: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_out['is_construction_routelevel'] = route_merged['is_construction_routelevel'].fillna(False).astype('int8')


/Users/proteeksanyal/Desktop/Learning/Transit-AI/notebooks/./enrich.py:263: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_out['is_maintenance_routelevel'] = route_merged['is_maintenance_routelevel'].fillna(False).astype('int8')


/Users/proteeksanyal/Desktop/Learning/Transit-AI/notebooks/./enrich.py:264: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_out['is_rail_replacement_routelevel'] = route_merged['is_rail_replacement_routelevel'].fillna(False).astype('int8')


/Users/proteeksanyal/Desktop/Learning/Transit-AI/notebooks/./enrich.py:269: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_out['is_construction_stoplevel'] = stop_merged['is_construction_stoplevel'].fillna(False).astype('int8')


/Users/proteeksanyal/Desktop/Learning/Transit-AI/notebooks/./enrich.py:270: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_out['is_maintenance_stoplevel'] = stop_merged['is_maintenance_stoplevel'].fillna(False).astype('int8')


/Users/proteeksanyal/Desktop/Learning/Transit-AI/notebooks/./enrich.py:271: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_out['is_rail_replacement_stoplevel'] = stop_merged['is_rail_replacement_stoplevel'].fillna(False).astype('int8')


Route-alert join match rate: 36.1440%
Stop-alert join match rate:  1.6884%


In [6]:
# Cell 6 — +both: weather and alerts attached to the same frame
X_both_full, _ = enrich.attach_weather(X_alerts_full, scheduled_arrival_dt, S3_BUCKET, fs)

Weather join match rate: 100.00%


## Leakage check on the alert join (measurement only — no fix applied here)

The alert tables are keyed by the hour an alert was **published**
(`route_hourly`/`stop_hourly`'s grain is literally "an alert was observed
in a snapshot during this hour"). Joining on the row's *scheduled-arrival*
hour means a match at `(alert_date, alert_hour)` implies an alert was
published during that specific hour — but if that hour is **later than**
the row's own `snapshot_timestamp` (i.e. later than when this observation
was actually captured / when the prediction would be made), the model is
being handed information about an alert that didn't exist yet at prediction
time. This is measured directly below: for every alert-matched row, compare
the matched hour's start against `snapshot_timestamp`.

In [7]:
# Cell 7 — Measure alert-join leakage (report only, no fix applied)
sched_hour_start = scheduled_arrival_dt.dt.floor('h')

is_alert_matched = (X_alerts_full['route_has_alert'] == 1) | (X_alerts_full['stop_has_alert'] == 1)
is_future_publish = sched_hour_start > snapshot_timestamp

leakage_mask = is_alert_matched & is_future_publish
n_matched = int(is_alert_matched.sum())
n_leak = int(leakage_mask.sum())
leak_pct = (n_leak / n_matched * 100) if n_matched else 0.0

print(f'Alert-matched rows (route or stop): {n_matched:,}')
print(f'Of those, matched-hour is AFTER snapshot_timestamp (potential leakage): {n_leak:,} ({leak_pct:.2f}%)')
print('\nNo fix applied — measurement only, per instructions. Decide next step based on this magnitude.')

Alert-matched rows (route or stop): 6,749,135
Of those, matched-hour is AFTER snapshot_timestamp (potential leakage): 3,528,076 (52.27%)

No fix applied — measurement only, per instructions. Decide next step based on this magnitude.


## Train four variants

In [8]:
# Cell 8 — Train all four variants with identical XGBRegressor params
XGB_PARAMS = dict(enable_categorical=True, tree_method='hist', random_state=42, n_jobs=-1)

variant_cols = {
    'baseline': feature_cols,
    '+weather': feature_cols + enrich.WEATHER_COLS,
    '+alerts': feature_cols + enrich.ALERT_COLS,
    '+both': feature_cols + enrich.WEATHER_COLS + enrich.ALERT_COLS,
}
variant_frames = {
    'baseline': X_base,
    '+weather': X_weather_full,
    '+alerts': X_alerts_full,
    '+both': X_both_full,
}

models = {}
preds = {}
for name, cols in variant_cols.items():
    Xv = variant_frames[name]
    X_train = Xv.loc[train_mask, cols].copy()
    X_test = Xv.loc[test_mask, cols].copy()

    model = xgb.XGBRegressor(**XGB_PARAMS)
    model.fit(X_train, y_train)
    preds[name] = model.predict(X_test)
    models[name] = model
    print(f'Trained {name}: {len(cols)} features -> {X_train.shape}')

Trained baseline: 8 features -> (15431112, 8)


Trained +weather: 14 features -> (15431112, 14)


Trained +alerts: 20 features -> (15431112, 20)


Trained +both: 26 features -> (15431112, 26)


## Report: comparison table

In [9]:
# Cell 9 — Comparison table: variant | test MAE | test RMSE | delta vs baseline | % improvement over naive-median
def mae(a, b):
    return float(np.mean(np.abs(a - b)))

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

train_median = float(y_train.median())
naive_pred = np.full(len(y_test), train_median)
naive_mae = mae(y_test.values, naive_pred)
naive_rmse = rmse(y_test.values, naive_pred)

rows = []
baseline_mae = None
for name in ['baseline', '+weather', '+alerts', '+both']:
    m = mae(y_test.values, preds[name])
    r = rmse(y_test.values, preds[name])
    if name == 'baseline':
        baseline_mae = m
    rows.append({
        'variant': name,
        'test_MAE': round(m, 4),
        'test_RMSE': round(r, 4),
        'delta_MAE_vs_baseline': round(m - baseline_mae, 4),
        'pct_improvement_over_naive': round((1 - m / naive_mae) * 100, 2),
    })

comparison_df = pd.DataFrame(rows)
print(f'Naive median baseline (train median={train_median:.3f} min): MAE={naive_mae:.3f}, RMSE={naive_rmse:.3f}\n')
print(comparison_df.to_string(index=False))

Naive median baseline (train median=0.283 min): MAE=2.845, RMSE=5.762

 variant  test_MAE  test_RMSE  delta_MAE_vs_baseline  pct_improvement_over_naive
baseline    2.2647     4.8412                 0.0000                       20.39
+weather    2.3231     4.9807                 0.0584                       18.34
 +alerts    2.2551     4.7302                -0.0096                       20.72
   +both    2.2801     4.9195                 0.0154                       19.85


## Report: gain importance of ONLY the added columns (variants 2-4)

In [10]:
# Cell 10 — Gain importance restricted to the columns each variant added over baseline
added_cols_map = {
    '+weather': enrich.WEATHER_COLS,
    '+alerts': enrich.ALERT_COLS,
    '+both': enrich.WEATHER_COLS + enrich.ALERT_COLS,
}

gain_importance = {}
for name, added_cols in added_cols_map.items():
    booster = models[name].get_booster()
    gain = booster.get_score(importance_type='gain')
    filtered = {k: round(v, 2) for k, v in gain.items() if k in added_cols}
    filtered = dict(sorted(filtered.items(), key=lambda kv: -kv[1]))
    gain_importance[name] = filtered

    print(f'--- {name}: gain importance of ADDED columns only ---')
    if filtered:
        for k, v in filtered.items():
            print(f'  {k:35s} {v:>10.2f}')
    else:
        print('  (none of the added columns were ever used in a split)')
    unused = [c for c in added_cols if c not in gain]
    if unused:
        print(f'  never used in any split: {unused}')
    print()

--- +weather: gain importance of ADDED columns only ---
  wind_gusts_10m                        15871.23
  temperature_2m                        15070.12
  wind_speed_10m                        13278.02
  precipitation                          5303.58
  never used in any split: ['is_raining', 'is_weather_missing']

--- +alerts: gain importance of ADDED columns only ---
  route_alert_effect                    12910.81
  route_n_alerts                         6224.52
  stop_alert_effect                      5287.16
  is_construction_routelevel             4586.04
  stop_has_alert                         1901.34
  route_has_alert                        1577.64
  never used in any split: ['is_maintenance_routelevel', 'is_rail_replacement_routelevel', 'stop_n_alerts', 'is_construction_stoplevel', 'is_maintenance_stoplevel', 'is_rail_replacement_stoplevel']

--- +both: gain importance of ADDED columns only ---
  is_construction_routelevel            32872.50
  wind_speed_10m                 

## Report: join match rates and alert sparsity

In [11]:
# Cell 11 — Join match rates (test set) and alert sparsity check
test_weather_match = float((1 - X_weather_full.loc[test_mask, 'is_weather_missing']).mean()) * 100
test_route_match = float(X_alerts_full.loc[test_mask, 'route_has_alert'].mean()) * 100
test_stop_match = float(X_alerts_full.loc[test_mask, 'stop_has_alert'].mean()) * 100
test_any_alert = float((
    (X_alerts_full.loc[test_mask, 'route_has_alert'] == 1) |
    (X_alerts_full.loc[test_mask, 'stop_has_alert'] == 1)
).mean()) * 100

print(f'Weather join match rate (test set):     {test_weather_match:.2f}%')
print(f'Route-alert join match rate (test set): {test_route_match:.4f}%')
print(f'Stop-alert join match rate (test set):  {test_stop_match:.4f}%')
print(f'% test rows with ANY alert (route or stop): {test_any_alert:.4f}%')

if test_any_alert < 1.0:
    print(
        f'\nAlert coverage is {test_any_alert:.4f}% of test rows — well under 1%. '
        'With only ~1,842 distinct alerts against tens of millions of ml_features rows, '
        'the alert columns are almost certainly too sparse for a tree ensemble to learn '
        'a reliable, generalizable split from directly. Any MAE movement from +alerts should '
        'be read with that sparsity in mind rather than as strong evidence either way.'
    )

Weather join match rate (test set):     100.00%
Route-alert join match rate (test set): 31.2208%
Stop-alert join match rate (test set):  1.1203%
% test rows with ANY alert (route or stop): 31.8588%


## Write JSON summary to S3

In [12]:
# Cell 12 — Write run summary to s3://<bucket>/ml_features/enrichment_ab/<run_date>.json
summary = {
    'generated_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'ml_features_run_date': ML_RUN_DATE,
    'boundary_date': str(boundary_date),
    'n_train': int(train_mask.sum()),
    'n_test': int(test_mask.sum()),
    'naive_median_train': train_median,
    'naive_median_mae': naive_mae,
    'naive_median_rmse': naive_rmse,
    'variants': comparison_df.to_dict(orient='records'),
    'gain_importance_added_columns': gain_importance,
    'join_match_rates_test': {
        'weather_pct': test_weather_match,
        'route_alert_pct': test_route_match,
        'stop_alert_pct': test_stop_match,
        'any_alert_pct': test_any_alert,
    },
    'alert_join_leakage_check': {
        'n_alert_matched_rows': n_matched,
        'n_future_publish_rows': n_leak,
        'pct_future_publish_of_matched': leak_pct,
        'note': 'Measured only — no fix applied. Alert published-hour matched row is later than '
                'the row\'s snapshot_timestamp for this % of alert-matched rows.',
    },
}

RUN_DATE_OUT = ML_RUN_DATE  # tie this artifact to the ml_features snapshot version it evaluated
summary_path = f's3://{S3_BUCKET}/ml_features/enrichment_ab/{RUN_DATE_OUT}.json'
with fs.open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f'Wrote enrichment A/B summary to {summary_path}')

Wrote enrichment A/B summary to s3://seq-transit-ai-data-ps/ml_features/enrichment_ab/2026-07-17.json
